### Volatility Targets

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Locate the processed stock data directory
possible_paths = [
    Path("notebooks/data/processed/stocks"),
    Path("data/processed/stocks"),
]

stock_dir = next(
    (path for path in possible_paths if path.exists()),
    None
)

if stock_dir is None:
    raise FileNotFoundError("Could not locate the processed stocks directory.")

# List available datasets
stock_files = sorted(stock_dir.glob("*.parquet"))

print(f"Processed stock directory: {stock_dir}")
print(f"Number of stock datasets: {len(stock_files)}")
print("\nFirst five files:")
for file in stock_files[:5]:
    print(file.name)

# inspecting one dataset
sample = pd.read_parquet(stock_files[0])

print("\nDataset shape:")
print(sample.shape)

print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

print("\nFirst five rows:")
display(sample.head())

Processed stock directory: data/processed/stocks
Number of stock datasets: 50

First five files:
ADANIENT_NS.parquet
ADANIPORTS_NS.parquet
APOLLOHOSP_NS.parquet
ASIANPAINT_NS.parquet
AXISBANK_NS.parquet

Dataset shape:
(4159, 8)

Columns:
['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']

Data types:
Open            float64
High            float64
Low             float64
Close           float64
Adj Close       float64
Volume            int64
Dividends       float64
Stock Splits    float64
dtype: object

First five rows:


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
Date,,,,,,,,
2010-01-04,67.011856,69.586876,66.368103,69.219017,57.527908,1860224,0.0,0.0
2010-01-05,69.433601,71.111961,69.433601,70.820740,58.859104,2280286,0.0,0.0
2010-01-06,70.966354,79.243202,70.506531,77.810081,64.667953,7326390,0.0,0.0
2010-01-07,79.611069,79.611069,74.031853,74.598969,61.999172,4759208,0.0,0.0
2010-01-08,75.074120,77.863731,73.909233,75.541611,62.782619,3324076,0.0,0.0


In [3]:
# Locate TMPV dataset
stock_dir = Path("data/processed/stocks")
tmpv_files = list(stock_dir.glob("TMPV*.parquet"))

if len(tmpv_files) != 1:
    raise FileNotFoundError(
        f"Expected one TMPV file, found: {[f.name for f in tmpv_files]}"
    )

tmpv = pd.read_parquet(tmpv_files[0]).copy()

# Ensure dates are sorted
tmpv.index = pd.to_datetime(tmpv.index)
tmpv = tmpv.sort_index()

# Inspect the period around the demerger
window = tmpv.loc["2025-10-07":"2025-10-21"].copy()

# Calculate close-to-close log returns
window["Close_Return_%"] = (
    np.log(window["Close"] / window["Close"].shift(1)) * 100
)

window["Adj_Close_Return_%"] = (
    np.log(window["Adj Close"] / window["Adj Close"].shift(1)) * 100
)

# Compare adjusted and unadjusted prices
window["Adj_Close_Ratio"] = (
    window["Adj Close"] / window["Close"]
)

display(
    window[
        [
            "Close",
            "Adj Close",
            "Dividends",
            "Stock Splits",
            "Close_Return_%",
            "Adj_Close_Return_%",
            "Adj_Close_Ratio",
        ]
    ].round(4)
)

,Close,Adj Close,Dividends,Stock Splits,Close_Return_%,Adj_Close_Return_%,Adj_Close_Ratio
Date,,,,,,,
2025-10-07,698.05,692.3071,0.0,0.0,NaN,NaN,0.9918
2025-10-08,681.55,675.9429,0.0,0.0,-2.3921,-2.3921,0.9918
2025-10-09,681.10,675.4965,0.0,0.0,-0.0660,-0.0661,0.9918
2025-10-10,678.95,673.3643,0.0,0.0,-0.3162,-0.3162,0.9918
2025-10-13,660.75,655.3140,0.0,0.0,-2.7172,-2.7172,0.9918
2025-10-14,395.45,392.1966,0.0,0.0,-51.3351,-51.3351,0.9918
2025-10-15,390.85,387.6345,0.0,0.0,-1.1701,-1.1701,0.9918
2025-10-16,396.80,393.5355,0.0,0.0,1.5108,1.5108,0.9918
2025-10-17,396.60,393.3372,0.0,0.0,-0.0504,-0.0504,0.9918


#### Calculate Returns and Forward Volatility Targets for all 50 Stocks
- Daily log returns:

rt = ln(Pt / Pt-1)

- Forward realized volatility proxy: to estimate how much a stock's returns fluctuate over a future period, starting after the prediction date



In [4]:
stock_dir = Path("data/processed/stocks")
target_dir = Path("data/processed/volatility_targets")

target_dir.mkdir(parents=True, exist_ok=True)

HORIZONS = [5, 21]

# Known TMPV demerger-related return
TMPV_INVALID_RETURN_DATE = pd.Timestamp("2025-10-14")

def calculate_volatility_targets(df, ticker):
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    # Basic data checks
    if df.index.has_duplicates:
        raise ValueError(f"{ticker}: Duplicate dates found.")

    if "Adj Close" not in df.columns:
        raise ValueError(f"{ticker}: Adj Close column is missing.")

    prices = df["Adj Close"]

    if (prices.dropna() <= 0).any():
        raise ValueError(f"{ticker}: Non-positive adjusted prices found.")

    # Daily close-to-close log returns
    returns = np.log(prices / prices.shift(1))

    # Exclude the known TMPV demerger-related return
    if ticker.startswith("TMPV"):
        if TMPV_INVALID_RETURN_DATE in returns.index:
            returns.loc[TMPV_INVALID_RETURN_DATE] = np.nan

    df["Daily_Return"] = returns

    # Squared returns form the volatility proxy
    squared_returns = returns.pow(2)

    for h in HORIZONS:

        # Sum the next h squared returns, excluding today's return
        future_squared_sum = (
            squared_returns.iloc[::-1]
            .rolling(window=h, min_periods=h)
            .sum()
            .shift(1)
            .iloc[::-1]
        )

        # Annualized forward realized-volatility proxy
        df[f"RV_{h}D"] = np.sqrt(
            (252 / h) * future_squared_sum
        )

    return df[
        ["Daily_Return", "RV_5D", "RV_21D"]
    ]



stock_files = sorted(stock_dir.glob("*.parquet"))

if len(stock_files) != 50:
    raise ValueError(
        f"Expected 50 stock files, found {len(stock_files)}."
    )

for file in stock_files:

    ticker = file.stem
    stock_data = pd.read_parquet(file)

    targets = calculate_volatility_targets(
        stock_data,
        ticker
    )

    output_path = target_dir / f"{ticker}_targets.parquet"
    targets.to_parquet(output_path)

print(f"Processed {len(stock_files)} stocks.")
print(f"Target files saved to: {target_dir}")

Processed 50 stocks.
Target files saved to: data/processed/volatility_targets


In [12]:
# Inspect one stock's targets
sample_file = sorted(target_dir.glob("*.parquet"))[0]

sample = pd.read_parquet(sample_file)

display(sample.head(10))
display(sample.tail(10))

# Check target availability
print("\nMissing values:")
display(sample.isna().sum())

print("\nTarget summary:")
display(sample[["RV_5D", "RV_21D"]].describe())


,Daily_Return,RV_5D,RV_21D
Date,,,
2010-01-04,NaN,0.765567,0.522710
2010-01-05,0.022876,0.890553,0.519890
2010-01-06,0.094119,0.594152,0.413778
2010-01-07,-0.042145,0.587979,0.389027
2010-01-08,0.012557,0.581184,0.389658
2010-01-11,0.017698,0.573791,0.385179
2010-01-12,-0.068046,0.310922,0.306408
2010-01-13,-0.011266,0.304700,0.304957
2010-01-14,0.040391,0.188332,0.277383


,Daily_Return,RV_5D,RV_21D
Date,,,
2026-10-26,0.012437,0.383523,NaN
2026-10-27,-0.007203,0.384843,NaN
2026-10-28,-0.004105,0.493560,NaN
2026-10-29,0.050367,0.345971,NaN
2026-10-30,-0.010988,0.338614,NaN
2026-11-02,0.013860,NaN,NaN
2026-11-03,-0.008486,NaN,NaN
2026-11-04,0.043722,NaN,NaN
2026-11-05,0.008856,NaN,NaN



Missing values:


Daily_Return     1
RV_5D            5
RV_21D          21
dtype: int64


Target summary:


,RV_5D,RV_21D
count,4154.000000,4138.000000
mean,0.408877,0.452001
std,0.305754,0.237854
min,0.013696,0.059060
25%,0.228408,0.310093
50%,0.344652,0.402776
75%,0.495451,0.515185
max,3.654108,2.042363


# Interpretation:
Daily_Return = 0.022876 means a 2.29% daily return.


RV_5D = 0.765567 means approximately 76.56% annualized volatility.

RV_21D = 0.452001 means approximately 45.20% annualized volatility for that observation.

In [14]:
# Validate volatility targets across all stocks

validation_results = []

for file in sorted(target_dir.glob("*_targets.parquet")):

    ticker = file.name.replace("_targets.parquet", "")
    targets = pd.read_parquet(file)
    original = pd.read_parquet(stock_dir / f"{ticker}.parquet")

    validation_results.append({
        "Ticker": ticker,
        "Dates Match": targets.index.equals(original.index),
        "RV_5D Boundary OK": targets["RV_5D"].tail(5).isna().all(),
        "RV_21D Boundary OK": targets["RV_21D"].tail(21).isna().all()
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

print("Stocks processed:", len(validation_df))
print("All checks passed:", validation_df.iloc[:, 1:].all().all())

# Verify TMPV demerger treatment
tmpv = pd.read_parquet(target_dir / "TMPV_NS_targets.parquet")

print(
    "TMPV demerger return excluded:",
    pd.isna(tmpv.loc["2025-10-14", "Daily_Return"])
)

,Ticker,Dates Match,RV_5D Boundary OK,RV_21D Boundary OK
0,ADANIENT_NS,True,True,True
1,ADANIPORTS_NS,True,True,True
2,APOLLOHOSP_NS,True,True,True
3,ASIANPAINT_NS,True,True,True
4,AXISBANK_NS,True,True,True
5,BAJAJ-AUTO_NS,True,True,True
6,BAJAJFINSV_NS,True,True,True
7,BAJFINANCE_NS,True,True,True
8,BEL_NS,True,True,True
9,BHARTIARTL_NS,True,True,True


Stocks processed: 50
All checks passed: True
TMPV demerger return excluded: True


### Lagged Returns and Historical Volatility

In [15]:
# Creating Lagged Return and Historical Volatility Features

feature_dir = Path("data/processed/features")
feature_dir.mkdir(parents=True, exist_ok=True)

return_lags = [1,2,5,10,21]
vol_windows = [5, 21, 63]

target_files = sorted(target_dir.glob("*_targets.parquet"))

if len(target_files) != 50:
    raise ValueError(f"Expected 50 target files, found {len(target_files)}")

for file in target_files:
    ticker = file.name.replace("_targets.parquet", "")

    # already validated daily returns
    targets = pd.read_parquet(file)
    returns = targets["Daily_Return"]

    features = pd.DataFrame(index=targets.index)

    # lagged daily returns
    for lag in return_lags:
        features[f"Return_Lag_{lag}"] = returns.shift(lag)

    # historical annualized volatility
    for window in vol_windows:
        historical_returns = returns.shift(1)

        features[f"Hist_Vol_{window}D"] = (
            historical_returns
            .rolling(window=window, min_periods=window)
            .std()
            * np.sqrt(252)
        )

    # save features separately
    features.to_parquet(feature_dir / f"{ticker}_features.parquet")

print(f"Created features for {len(target_files)} stocks.")
print(f"Saved to: {feature_dir}")


Created features for 50 stocks.
Saved to: data/processed/features


In [20]:
# Inspect one stock's engineered features

sample_file = sorted(feature_dir.glob("*_features.parquet"))[0]
sample_features = pd.read_parquet(sample_file)

display(sample_features.head(10).round(4))

print("Number of stocks:", len(list(feature_dir.glob("*_features.parquet"))))
print("Number of features:", sample_features.shape[1])
print("Missing values:")
display(sample_features.isna().sum())


,Return_Lag_1,Return_Lag_2,Return_Lag_5,Return_Lag_10,Return_Lag_21,Hist_Vol_5D,Hist_Vol_21D,Hist_Vol_63D
Date,,,,,,,,
2010-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,0.0229,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,0.0941,0.0229,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,-0.0421,0.0941,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-11,0.0126,-0.0421,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-12,0.0177,0.0126,0.0229,NaN,NaN,0.7703,NaN,NaN
2010-01-13,-0.0680,0.0177,0.0941,NaN,NaN,0.9944,NaN,NaN
2010-01-14,-0.0113,-0.0680,-0.0421,NaN,NaN,0.5801,NaN,NaN


Number of stocks: 50
Number of features: 8
Missing values:


Return_Lag_1      2
Return_Lag_2      3
Return_Lag_5      6
Return_Lag_10    11
Return_Lag_21    22
Hist_Vol_5D       6
Hist_Vol_21D     22
Hist_Vol_63D     64
dtype: int64

In [22]:
# Verify when each feature first becomes available

display(
    sample_features.apply(lambda col: col.first_valid_index())
)

Return_Lag_1    2010-01-06
Return_Lag_2    2010-01-07
Return_Lag_5    2010-01-12
Return_Lag_10   2010-01-19
Return_Lag_21   2010-02-04
Hist_Vol_5D     2010-01-12
Hist_Vol_21D    2010-02-04
Hist_Vol_63D    2010-04-09
dtype: datetime64[ms]

## Volume-based features


In [24]:
# Create Volume Based Features

for file in sorted(target_dir.glob("*_targets.parquet")):

    ticker = file.name.replace("_targets.parquet", "")

    stock = pd.read_parquet(stock_dir / f"{ticker}.parquet")
    features = pd.read_parquet(feature_dir / f"{ticker}_features.parquet")

    volume = stock["Volume"]

    # Daily percentage change in volume
    features["Volume_Change_1D"] = volume.pct_change()

    # Current volume relative to previous average volume
    for window in [5, 21]:
        previous_avg = volume.shift(1).rolling(
            window=window,
            min_periods=window
        ).mean()

        features[f"Volume_MA_Ratio_{window}D"] = (
            volume / previous_avg
        )

    features.to_parquet(feature_dir / f"{ticker}_features.parquet")

In [25]:
display(
    features[
        [
            "Volume_Change_1D",
            "Volume_MA_Ratio_5D",
            "Volume_MA_Ratio_21D"
        ]
    ].head(30).round(3)
)

,Volume_Change_1D,Volume_MA_Ratio_5D,Volume_MA_Ratio_21D
Date,,,
2010-01-04,NaN,NaN,NaN
2010-01-05,0.237,NaN,NaN
2010-01-06,-0.144,NaN,NaN
2010-01-07,-0.458,NaN,NaN
2010-01-08,0.809,NaN,NaN
2010-01-11,-0.283,0.758,NaN
2010-01-12,-0.522,0.382,NaN
2010-01-13,1.114,0.997,NaN
2010-01-14,1.698,2.928,NaN
